In [1]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.integrate import quad
from scipy.sparse.linalg import eigsh
from scipy.sparse import diags
from scipy.interpolate import UnivariateSpline
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

# =============================================
# 1. Parámetros Físicos y Configuración
# =============================================
D0 = 2e-9           # Coeficiente de difusión base [m²/s]
R = 1e-6            # Radio del cilindro [m]
z_c = 10e-9         # Altura crítica [m]
lambda_param = 0.2  # Parámetro de decaimiento [nm⁻¹]
kappa = 0.0         # Tasa de adsorción superficial [s⁻¹]
rho0 = 0.75e-6      # Posición radial inicial [m]
z0 = 15e-9          # Posición axial inicial [m]
n_modes_radial = 10 # Modos radiales
m_max = 10          # Modos axiales
N_z = 800           # Nodos en z
alpha_grid = 10.0   # Parámetro de densidad de malla
delta = 1.0e-9      # Ancho de transición
N_quad = 30         # Puntos de cuadratura por eje

# =============================================
# 2. Generación de Malla Adaptativa
# =============================================
j = np.arange(N_z)
z = z_c * (1 + np.sinh(alpha_grid*(j/(N_z-1) - 0.5*alpha_grid) / np.sinh(0.5*alpha_grid))
dz = np.diff(z)

# =============================================
# 3. Perfil de Difusión D(z) CORREGIDO
# =============================================
def D(z):
    S = 0.5*(1 + np.tanh((z - z_c)/delta))
    return D0*(np.exp(-lambda_param*z)*(1 - S) + S  # Corrección clave

D_values = D(z)

# =============================================
# 4. Solución Radial Vectorizada
# =============================================
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros/R

# Cálculo de normas usando cuadratura vectorizada
rho_quad = np.linspace(0.01*R, 0.99*R, 1000)
norm_J0_sq = np.array([np.trapz(rho_quad * jv(0, alpha_n[n]*rho_quad)**2, rho_quad)
                for n in range(n_modes_radial)])

# =============================================
# 5. Solución Axial Paralelizada
# =============================================
def solve_axial_mode(n):
    main_diag = np.zeros(N_z)
    upper_diag = np.zeros(N_z-1)
    lower_diag = np.zeros(N_z-1)
    
    # Construcción de matriz optimizada
    for j in range(1, N_z-1):
        h_prev = dz[j-1]
        h_next = dz[j]
        D_prev = 0.5*(D_values[j] + D_values[j-1])
        D_next = 0.5*(D_values[j] + D_values[j+1])
        
        main_diag[j] = D_prev/h_prev + D_next/h_next + alpha_n[n]**2*D_values[j]
        upper_diag[j] = -D_next/h_next
        lower_diag[j-1] = -D_prev/h_prev

    # Condiciones de frontera
    h0 = dz[0]
    if kappa == 0:
        main_diag[0] = 1.0
        upper_diag[0] = -1.0  # Z0 = Z1
    else:
        main_diag[0] = D_values[0]/h0 + kappa
        upper_diag[0] = -D_values[0]/h0
    
    main_diag[-1] = 1.0
    lower_diag[-1] = -1.0
    
    A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1], format='csr')
    eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SA', tol=1e-8)
    return eigenvalues, eigenvectors

# Ejecución paralela
results = Parallel(n_jobs=-1)(delayed(solve_axial_mode)(n) for n in range(n_modes_radial))
eigenvalues_dict = {n: res[0] for n, res in enumerate(results)}
eigenvectors_dict = {n: res[1] for n, res in enumerate(results)}

# =============================================
# 6. Precomputación de Interpolantes
# =============================================
Z_interps = [[UnivariateSpline(z, eigenvectors_dict[n][:,m], s=0) 
            for m in range(m_max)] for n in range(n_modes_radial)]

# =============================================
# 7. Cálculo de Coeficientes A_nm Vectorizado
# =============================================
def compute_A_nm(n):
    # Cálculo vectorizado de normas
    z_quad = np.linspace(z[0], z[-1], 1000)
    Z_norms = np.array([
        np.trapz(Z_interps[n][m](z_quad)**2 / D(z_quad), z_quad)
        for m in range(m_max)
    ])
    
    Z_m_z0 = np.array([Z_interps[n][m](z0) for m in range(m_max)])
    J0_rho0 = jv(0, alpha_n[n]*rho0)
    
    valid_modes = (Z_norms > 1e-25) & (np.abs(Z_m_z0) > 1e-12)
    return (J0_rho0 * Z_m_z0 * valid_modes) / (2*np.pi * rho0 * norm_J0_sq[n] * (Z_norms + 1e-30))

A_nm = Parallel(n_jobs=-1)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))

# =============================================
# 8. Cálculo de G(t) Vectorizado
# =============================================
# Configuración de cuadratura vectorizada
rho_nodes, rho_weights = np.polynomial.legendre.leggauss(N_quad)
rho_nodes = 0.1*R + 0.8*R*(rho_nodes + 1)/2
rho_weights *= 0.8*R

z_nodes, z_weights = np.polynomial.legendre.leggauss(N_quad)
z_nodes = z_c - 5e-9 + 10e-9*(z_nodes + 1)/2
z_weights *= 10e-9

# Crear malla vectorizada
Rg, Zg = np.meshgrid(rho_nodes, z_nodes)
R_flat = Rg.flatten()
Z_flat = Zg.flatten()
weights = (rho_weights[:, None] * z_weights[None, :]).flatten()

# Precalcular términos geométricos
r_mag = np.sqrt(R_flat**2 + Z_flat**2)
cos_theta = Z_flat / (r_mag + 1e-30)
f_values = (3*cos_theta**2 - 1) / (r_mag**3 + 1e-30)

def compute_G(t):
    total = 0.0
    for n in range(n_modes_radial):
        # Evaluación vectorizada para todos los puntos
        J0 = jv(0, alpha_n[n]*R_flat)
        decay = np.exp(-(alpha_n[n]**2*D0 + eigenvalues_dict[n][0])*t)
        Z_m = Z_interps[n][0](Z_flat)
        
        # Contribución vectorizada del modo
        contrib = A_nm[n][0] * J0 * Z_m * decay
        total += np.sum(weights * f_values * contrib)
    
    return (5/(16*np.pi)) * np.clip(total, 1e-40, 1e10)

# =============================================
# 9. Visualización y Ejecución
# =============================================
if __name__ == "__main__":
    # Cálculo de G(t)
    t_values = np.logspace(-9, -3, 50)
    G_values = Parallel(n_jobs=-1)(delayed(compute_G)(t) for t in t_values)
    
    # Guardado de datos
    np.savetxt('Gt_results.dat', np.column_stack((t_values, G_values)),
               header='Time(s) G(t)', fmt='%.6e')
    
    # Gráfico
    plt.figure(figsize=(10, 6))
    plt.loglog(t_values, G_values, 'o-', label='Simulación')
    plt.xlabel('Tiempo (s)')
    plt.ylabel('G(t)')
    plt.title('Función de Correlación Nano-NMR')
    plt.grid(True, which='both', ls='--')
    plt.savefig('Gt_result.pdf')
    plt.close()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3682664090.py, line 31)